# 02 — Matching Analysis

Run the cross-platform matcher and inspect the candidate pairs.


In [ ]:
import asyncio, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from clients.kalshi_client import KalshiClient
from clients.polymarket_client import PolymarketClient
from core.event_matcher import EventMatcher, flatten_market_pairs
from core.market_matcher import MatchCache
from core.fee_calculator import polymarket_fee_per_contract, kalshi_fee_per_contract
from data.models import Platform
import pandas as pd


In [ ]:
KALSHI_SERIES = [
    "KXPRES", "KXSEN", "KXBTCD", "KXETHD", "KXBTC", "KXETH",
    "KXFEDDECISION", "KXCPI", "KXJOBS", "KXSPX", "KXNASDAQ",
    "KXWORLDCUP", "KXNBA", "KXWAR",
]
k = KalshiClient(environment="production")
p = PolymarketClient()
k_events, p_events = await asyncio.gather(
    k.get_events_with_markets(KALSHI_SERIES, limit_per_series=30),
    p.get_events_with_markets(limit=300),
)
print(f"Events: K={len(k_events)}  P={len(p_events)}")
print(f"Total nested markets: K={sum(len(e.markets) for e in k_events)}  "
      f"P={sum(len(e.markets) for e in p_events)}")


In [ ]:
matcher = EventMatcher(min_event_score=0.45, strike_tolerance_pct=0.02)
event_pairs = matcher.match(k_events, p_events)
pairs = flatten_market_pairs(event_pairs)
print(f"Event matches: {len(event_pairs)}")
print(f"Market pairs:  {len(pairs)}")


In [ ]:
# Event-level pairings
ep_rows = []
for ep in event_pairs:
    ep_rows.append({
        "conf": round(ep.confidence, 3),
        "method": ep.method.value,
        "kalshi_event": ep.kalshi_event.event_id[:35],
        "kalshi_title": ep.kalshi_event.title[:55],
        "poly_event": ep.polymarket_event.event_id[:35],
        "poly_title": ep.polymarket_event.title[:55],
        "n_market_pairs": len(ep.market_pairs),
        "end_delta_hrs": round(ep.end_date_delta_hours, 1),
    })
pd.DataFrame(ep_rows).sort_values("conf", ascending=False)


In [ ]:
# Market-level pairings within events
rows = []
for pp in pairs:
    rows.append({
        "conf": round(pp.confidence, 3),
        "method": pp.method.value,
        "k_outcome": (pp.kalshi_market.raw.get('yes_sub_title','') or '')[:30],
        "p_outcome": (pp.polymarket_market.raw.get('groupItemTitle','') or '')[:30],
        "k_strike":  pp.kalshi_market.raw.get('floor_strike'),
        "p_threshold": pp.polymarket_market.raw.get('groupItemThreshold'),
        "kalshi_id": pp.kalshi_market.market_id[:32],
    })
pd.DataFrame(rows).sort_values("conf", ascending=False).head(50)


## Mark verified / rejected matches

Review the table above. For pairs that ARE the same event, mark verified. For pairs that AREN'T, mark rejected. Both go into the cache for future runs.


In [ ]:
# Example: mark first match as verified
cache = MatchCache(path="../data/cache/matches_cache.db")
# cache.mark_verified("KXBTCD-26MAY1812-T76000", "0xabc...")
# cache.mark_rejected("KXSEN-26-AZ-D", "wrong-poly-condition-id")
print(f"Verified cache size: {len(cache.get_verified())}")
print(f"Rejected cache size: {len(cache.get_rejected())}")


## Pull live prices for top-confidence pairs and check for ARB right now


In [ ]:
top = pairs[:15]
prices = {}
for pp in top:
    try:
        kp, pp_pr = await asyncio.gather(
            k.get_price(pp.kalshi_market.market_id),
            p.get_price(pp.polymarket_market.market_id),
            return_exceptions=True,
        )
        if not isinstance(kp, Exception):
            prices[(Platform.KALSHI, pp.kalshi_market.market_id)] = kp
        if not isinstance(pp_pr, Exception):
            prices[(Platform.POLYMARKET, pp.polymarket_market.market_id)] = pp_pr
    except Exception as e:
        print(f"  err: {e}")
print(f"got prices for {len(prices)//2} pairs")


In [ ]:
from core.arb_detector import detect_all
ops = detect_all(top, prices, min_net_edge_cents=-10, min_liquidity_usd=0, max_capital_usd=5000)
print(f"{len(ops)} arb candidates (including negative-edge for visibility)")
arb_rows = []
for op in ops:
    arb_rows.append({
        "direction": op.direction.value,
        "k_title": op.pair.kalshi_market.title[:50],
        "k_price": round(op.kalshi_price, 4),
        "p_price": round(op.polymarket_price, 4),
        "gross_c": round(op.gross_edge_cents, 2),
        "net_c": round(op.net_edge_cents, 2),
        "size": round(op.max_size_contracts, 0),
        "capital": round(op.capital_required_usd, 0),
        "ann_ret%": round(op.annualized_return_pct or 0, 1),
    })
pd.DataFrame(arb_rows)


In [ ]:
await k.close(); await p.close()
